# Практика. RAG + RAGAS: multilingual E5, три LLM и резервные эталоны

**Роли строго разделены:**

- **Модель-ответчик (ANSWER)** отвечает пользователю по найденным чанкам. Именно её промпт изменяют участники.
- **Модель-генератор (GENERATOR)** создаёт вопросы и эталонные ответы через `RAGAS TestsetGenerator` (только в режиме `generate`).
- **LLM-судья (JUDGE)** независимо оценивает ответы модели-ответчика с помощью RAGAS.

Для генератора и судьи по умолчанию выбрана более мощная модель, чем для ответа. **Это настройки, а не гарантия объективности:** судья и эталоны тоже могут ошибаться.

В начале можно выбрать **один из трёх источников тестов**:

1. `generate` — RAGAS сам генерирует вопросы и эталоны из TXT.
2. `upload` — загрузка вручную проверенного CSV с вопросами и эталонами (запускается без генератора).
3. `example` — 18 готовых эталонов встроены в ноутбук; ничего загружать не нужно.

**Оценка RAGAS и ответы RAG выполняются на одних и тех же вопросах для обоих промптов.**

**Поиск:** вместо TF-IDF используется локальная эмбеддинг-модель `intfloat/multilingual-e5-small`. Она переводит вопрос и фрагменты справочника в векторы и ищет их по семантической близости. Для E5 обязательно добавляем `query:` к вопросу и `passage:` к текстам. Поиск не требует отдельного API-ключа или GPU.


## 0. Настройки — изменяйте сначала эту ячейку

Провайдеры выбираются **независимо** (`openai` / `gigachat`). Можно использовать один API-ключ для двух ролей, но сами объекты моделей и их настройки раздельны. Для первого запуска выберите `TESTSET_MODE = "example"`, чтобы не ждать генерации. Когда всё заработает, попробуйте `generate`.

При выборе `upload` появится окно для загрузки `ragas_manual_references_18.csv`. Формат: обязательные столбцы `user_input`, `reference`; дополнительные — `id`, `answerable`, `expected_section`.

In [ ]:
# ====== ГЛАВНЫЕ НАСТРОЙКИ ======
TESTSET_MODE = "generate"   # "generate" | "upload" | "example"

ANSWER_PROVIDER = "openai"   # "openai" | "gigachat"
ANSWER_MODEL = "gpt-4.1-mini"    # более доступная модель-ответчик

GENERATOR_PROVIDER = "openai"
GENERATOR_MODEL = "gpt-4.1"      # более сильная: создаёт тесты/эталоны

JUDGE_PROVIDER = "openai"
JUDGE_MODEL = "gpt-4.1"          # более сильная: судит ответы

EMBEDDING_PROVIDER = "openai"   # "openai" | "gigachat"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
GIGACHAT_SCOPE = "GIGACHAT_API_PERS"

TESTSET_SIZE = 6               # сколько новых вопросов сгенерирует RAGAS
REGENERATE_TESTSET = False     # не менять набор при повторных запусках
RUN_N = 3                      # сначала 3 вопроса; None = все
TOP_K = 3                      # число найденных чанков (одинаково для промптов)
E5_MODEL = "intfloat/multilingual-e5-small"  # русский язык; можно попробовать multilingual-e5-base
INCLUDE_EXTRA_METRICS = False  # Noise Sensitivity: медленно и дороже

print(f"Ответчик: {ANSWER_PROVIDER}/{ANSWER_MODEL}")
print(f"Генератор: {GENERATOR_PROVIDER}/{GENERATOR_MODEL}")
print(f"Судья: {JUDGE_PROVIDER}/{JUDGE_MODEL}")
print(f"Источник тестов: {TESTSET_MODE}")
print(f"Поиск RAG: {E5_MODEL}")

## 1. Установка и импорт библиотек

После **первой** установки в Colab желательно выполнить **Среда выполнения → Перезапустить сеанс**, а затем запустить ячейки с начала.

In [ ]:
# Важно: для первого запуска после установки перезапустите сеанс Colab.
# Sentence Transformers загружает E5 локально; RAGAS использует отдельные
# embeddings для своей генерации тестов и метрик.
%pip install -q "ragas==0.4.1" "langchain-community==0.4.1" "langchain-openai>=0.3" "langchain-gigachat>=0.3" "sentence-transformers>=3.3,<6" pandas matplotlib


In [ ]:
# Проверка, что библиотеки импортируются без ошибки VertexAI.
import importlib.metadata as md
print("RAGAS:", md.version("ragas"))
print("LangChain Community:", md.version("langchain-community"))
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
print("Импорт RAGAS прошёл успешно!")


## 2. Создаём три независимых модели

Секреты запрашиваются только для реально используемых провайдеров. В режиме `upload` и `example` модель **генератора не создаётся и не вызывается**.

**Два разных назначения embeddings:**
- **Локальная multilingual E5** — выбирает фрагменты из TXT для RAG. Никакого платного embedding API для поиска не требуется.
- **`embedding_model` (OpenAI/GigaChat)** — используется **внутри RAGAS** при синтезе тестов и расчёте отдельных метрик, например Semantic Similarity. Эта настройка остаётся независимой от E5.


In [ ]:
import os
from getpass import getpass

valid_providers = {"openai", "gigachat"}
providers_needed = {ANSWER_PROVIDER, JUDGE_PROVIDER, EMBEDDING_PROVIDER}
if TESTSET_MODE == "generate":
    providers_needed.add(GENERATOR_PROVIDER)
assert providers_needed <= valid_providers, "Провайдер должен быть openai или gigachat"
assert TESTSET_MODE in {"generate", "upload", "example"}

if "openai" in providers_needed:
    key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
    if not key: raise ValueError("Не задан ключ OpenAI")
    os.environ["OPENAI_API_KEY"] = key
if "gigachat" in providers_needed:
    key = os.environ.get("GIGACHAT_CREDENTIALS") or getpass("GigaChat credentials: ")
    if not key: raise ValueError("Не заданы credentials GigaChat")
    os.environ["GIGACHAT_CREDENTIALS"] = key

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_gigachat import GigaChat, GigaChatEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


def make_chat_model(provider, model_name):
    # Отдельный экземпляр модели для каждой роли.
    if provider == "openai":
        return ChatOpenAI(model=model_name, temperature=0)
    if provider == "gigachat":
        return GigaChat(
            credentials=os.environ["GIGACHAT_CREDENTIALS"],
            scope=GIGACHAT_SCOPE, model=model_name, temperature=0,
        )
    raise ValueError("Неизвестный провайдер")


def make_embedding_model(provider):
    if provider == "openai":
        return OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
    if provider == "gigachat":
        return GigaChatEmbeddings(
            credentials=os.environ["GIGACHAT_CREDENTIALS"], scope=GIGACHAT_SCOPE,
        )
    raise ValueError("Неизвестный провайдер для embeddings")

answer_model = make_chat_model(ANSWER_PROVIDER, ANSWER_MODEL)
judge_model = make_chat_model(JUDGE_PROVIDER, JUDGE_MODEL)
generator_model = (make_chat_model(GENERATOR_PROVIDER, GENERATOR_MODEL)
                   if TESTSET_MODE == "generate" else None)

embedding_model = make_embedding_model(EMBEDDING_PROVIDER)
judge_ragas_llm = LangchainLLMWrapper(judge_model)
generator_ragas_llm = (LangchainLLMWrapper(generator_model)
                       if generator_model is not None else None)
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

print("Модели и эмбеддинги подключены. Эталоны НЕ передаются модели-ответчику.")

## 3. Резервные эталоны: при желании загрузите CSV прямо сейчас

Выберите `TESTSET_MODE = "upload"` в верхних настройках, затем загрузите файл `ragas_manual_references_18.csv` **в этой ячейке**.

`example` использует точно такой же встроенный набор без загрузки. В `generate` вопросы будут получены из RAGAS. Режимы `upload` и `example` позволяют обойти **слабый/падающий генератор**, но **не устраняют погрешности LLM-судьи**.

In [ ]:
from pathlib import Path

UPLOAD_CSV_PATH = None
if TESTSET_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()   # выбираем файл с проверенными вопросами и эталонами
    if not uploaded:
        raise ValueError("Файл с эталонами не загружен")
    UPLOAD_CSV_PATH = Path(next(iter(uploaded)))
    print("Загружены эталоны:", UPLOAD_CSV_PATH)
elif TESTSET_MODE == "example":
    print("Будет использован встроенный ручной набор из 18 тестов")
else:
    print("Включена автоматическая генерация тестов RAGAS")

## 4. Подключаем TXT — источник истины для RAG

В учебном примере это вымышленный регламент «Люмен Лаб». Текст встроен в ноутбук и сохраняется как файл. При смене TXT нужно загружать **подходящие к нему эталоны**, иначе сравнение некорректно.

In [ ]:
from pathlib import Path

USE_OWN_TXT = False
TXT_PATH = Path("rag_knowledge_base.txt")
DEFAULT_TXT = '# Внутренний справочник компании «Люмен Лаб» (учебный пример)\n\nЭТО ВЫМЫШЛЕННАЯ КОМПАНИЯ И ВЫМЫШЛЕННЫЕ ПРАВИЛА. Все даты, суммы, названия и показатели придуманы специально для демонстрации RAG. Источником истины в этой лабораторной работе является только данный TXT-документ.\n\n## 01. Что делает компания\nКомпания «Люмен Лаб» разрабатывает платформу «Маяк» для мониторинга температуры и влажности на складах продуктов питания. Продукт не управляет холодильными установками, а только собирает показания датчиков, показывает графики и отправляет уведомления о превышении порогов. Пилотный запуск платформы «Маяк» состоялся 12 марта 2025 года.\n\n## 02. Ответственные за продукт\nРуководитель продукта «Маяк» — Алина Котова. За техническую инфраструктуру отвечает инженер Михаил Титов. Заявки на подключение клиентов принимает координатор Елена Левина. Все три роли относятся к продукту «Маяк», а не ко всей компании «Люмен Лаб».\n\n## 03. Тарифы\nТариф «Старт» стоит 1 400 рублей в месяц за один склад и включает до 20 датчиков. Тариф «Команда» стоит 3 900 рублей в месяц за один склад и включает до 100 датчиков. Единовременная стоимость подключения одного склада составляет 6 000 рублей независимо от тарифа. Оплата производится помесячно.\n\n## 04. Сбор данных\nВ стандартном режиме «Маяк» получает показания датчиков каждые 15 минут. В режиме критического наблюдения интервал уменьшается до 2 минут. Изменить режим может администратор склада. Если датчик перестал передавать показания более 30 минут, система создаёт уведомление «Нет связи с датчиком».\n\n## 05. Уведомления\nУведомления о превышении температуры отправляются по электронной почте и в Telegram. SMS-уведомления в текущей версии не поддерживаются. Пользователь может включить уведомления только для конкретного склада, но отдельная настройка на уровне конкретного датчика пока недоступна.\n\n## 06. Поддержка и инциденты\nОбычная техническая поддержка работает по будням с 09:00 до 18:00 по московскому времени. Критические инциденты уровня P1 принимаются круглосуточно. Для P1 первое подтверждение обращения должно поступить в течение 10 минут; целевое время решения — 4 часа. Для P2 первое подтверждение — в течение 1 часа; целевое время решения — до конца следующего рабочего дня.\n\n## 07. Безопасность и журналы\nЖурналы пользовательских действий хранятся 90 дней. Резервное копирование базы данных запускается ежедневно в 02:00 по московскому времени. Доступ к административной панели требует двухфакторной аутентификации. Документ не устанавливает срок хранения резервных копий.\n\n## 08. Интеграции и экспорт\n«Маяк» позволяет выгружать измерения в CSV и получать данные через REST API. Прямой экспорт в XLSX отсутствует. REST API доступен только на тарифе «Команда»; выгрузка CSV доступна на обоих тарифах.\n\n## 09. Доступность и обслуживание\nЦелевой показатель доступности платформы составляет 99,5% в месяц. Плановое техническое обслуживание предусмотрено по четвергам с 21:00 до 22:00 по московскому времени. Уведомление о плановом обслуживании публикуется за 24 часа до начала.\n\n## 10. Подключение нового клиента\nПодключение начинается с проверки списка датчиков, затем проводится тест передачи данных. Стандартное подключение одного склада занимает до 3 рабочих дней после получения корректного списка устройств. Обучающий вебинар для сотрудников клиента длится 90 минут.\n\n## 11. Пилотные клиенты\nК концу июня 2025 года к пилоту «Маяка» подключили 7 складов в трёх регионах. Конкретные названия регионов во внутреннем справочнике не указаны. Число складов не равно числу компаний-клиентов: один клиент может иметь несколько складов.\n\n## 12. Технологии\nВеб-API сервиса разработан на Python с использованием FastAPI. Основная операционная база данных — PostgreSQL. Для фоновых задач используется очередь Redis. Мобильное приложение в учебной версии платформы не предусмотрено.\n\n## 13. Обработка заявок и доступы\nНовый аккаунт администратора склада создаёт координатор после согласования с руководителем продукта. Доступ сотрудников к чужим складам по умолчанию закрыт. При увольнении сотрудника администратор склада должен отключить его учётную запись в течение одного рабочего дня.\n\n## 14. Ограничения справочника\nВ документе НЕТ информации о выручке компании, заработной плате сотрудников, банковских реквизитах, юридическом адресе, телефонном номере технической поддержки, планах выхода на зарубежные рынки и названии генерального директора. Нельзя делать выводы об этих сведениях на основе косвенных данных.\n'

if USE_OWN_TXT:
    from google.colab import files
    uploaded = files.upload()
    TXT_PATH = Path(next(iter(uploaded)))
elif not TXT_PATH.exists():
    TXT_PATH.write_text(DEFAULT_TXT, encoding="utf-8")

knowledge_base = TXT_PATH.read_text(encoding="utf-8")
print("База знаний:", TXT_PATH, "|", len(knowledge_base), "символов")
print(knowledge_base[:650], "...")

## 5. Semantic retrieval: multilingual E5

Используем **`intfloat/multilingual-e5-small`** — мультиязычную embedding-модель, которую скачиваем один раз с Hugging Face. Она работает с русскими запросами на CPU (GPU необязателен).

Алгоритм: **TXT → чанки → `passage:` → E5-векторы → `query:` + вопрос → E5-вектор → TOP-K по косинусному сходству**.

> У E5 префиксы `query:` и `passage:` важны для качества поиска. Эмбеддинги нормализуются, поэтому их скалярное произведение равно косинусному сходству. Индекс строится **один раз**: обе версии промпта получают одни и те же найденные чанки. У E5 ограничение длины входа (до 512 токенов), поэтому слишком длинные разделы дополнительно разрезаем.

[Модель E5 и инструкция по префиксам](https://huggingface.co/intfloat/multilingual-e5-small)


In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

# Разбиваем TXT по заголовкам. Длинные разделы разрезаем с перекрытием.
sections = [x.strip() for x in re.split(r"(?=^## )", knowledge_base, flags=re.M)
            if x.strip().startswith("## ")]
if not sections:                    # Собственный TXT без заголовков
    sections = [knowledge_base.strip()]
chunks = []
for section in sections:
    if len(section) <= 1000:
        chunks.append(section)
    else:
        for start in range(0, len(section), 850):
            piece = section[start:start + 1000].strip()
            if piece:
                chunks.append(piece)

if not chunks:
    raise ValueError("TXT-файл пустой — искать нечего")

# Скачается при первом запуске. Модель запускается локально в Colab.
e5 = SentenceTransformer(E5_MODEL)
print("E5 запущена на:", e5.device)

# Префиксы query: / passage: обязательны для корректной работы E5.
# Нормализация позволяет сравнивать векторы обычным скалярным произведением.
doc_vectors = e5.encode(
    ["passage: " + chunk for chunk in chunks],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

def retrieve(question, top_k=TOP_K):
    question_vector = e5.encode(
        ["query: " + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]
    similarities = doc_vectors @ question_vector   # cosine similarity
    best_ids = np.argsort(similarities)[::-1][:top_k]
    return [chunks[int(i)] for i in best_ids]

print("Чанков:", len(chunks))
print("Размерность E5:", doc_vectors.shape[1])
print("Пример поиска:")
for chunk in retrieve("Сколько хранятся журналы действий пользователей?"):
    print(" -", chunk.splitlines()[0][:110])


# Часть 1. Получаем тестовый набор

- В режиме **`generate`** RAGAS через **генератор** создаёт вопросы и эталоны из TXT, затем сохраняет их в CSV.
- В режиме **`upload`** берём вопросы и эталоны из выбранного CSV.
- В режиме **`example`** используем **встроенный проверенный набор** (14 вопросов с ответом + 4 без ответа).

**Один и тот же тестовый набор** используется для обеих версий промпта. Судья к его генерации отношения не имеет.

In [ ]:
# Только справочные тесты. Эти строки НЕ подаются модели-ответчику при генерации ответа.
# Если изменили исходный TXT, замените и эталоны.
BUILTIN_EXAMPLES = [
  {
    "id": 1,
    "user_input": "Как называется платформа и для чего она нужна?",
    "reference": "Платформа «Маяк» предназначена для мониторинга температуры и влажности на складах продуктов питания. Она не управляет холодильными установками.",
    "answerable": True,
    "expected_section": "01"
  },
  {
    "id": 2,
    "user_input": "Когда состоялся пилотный запуск «Маяка»?",
    "reference": "12 марта 2025 года.",
    "answerable": True,
    "expected_section": "01"
  },
  {
    "id": 3,
    "user_input": "Кто руководит продуктом «Маяк» и кто отвечает за инфраструктуру?",
    "reference": "Руководитель продукта — Алина Котова; за техническую инфраструктуру отвечает Михаил Титов.",
    "answerable": True,
    "expected_section": "02"
  },
  {
    "id": 4,
    "user_input": "Сколько стоит тариф «Старт» и сколько датчиков в нём доступно?",
    "reference": "1 400 рублей в месяц за один склад; до 20 датчиков.",
    "answerable": True,
    "expected_section": "03"
  },
  {
    "id": 5,
    "user_input": "Сколько стоит подключить один склад на тарифе «Команда»?",
    "reference": "Единовременное подключение стоит 6 000 рублей независимо от тарифа.",
    "answerable": True,
    "expected_section": "03"
  },
  {
    "id": 6,
    "user_input": "Как часто поступают показания в стандартном и критическом режимах?",
    "reference": "В стандартном режиме каждые 15 минут, в критическом каждые 2 минуты.",
    "answerable": True,
    "expected_section": "04"
  },
  {
    "id": 7,
    "user_input": "Какими каналами отправляются уведомления о температуре? Поддерживаются ли SMS?",
    "reference": "Уведомления отправляются по электронной почте и в Telegram; SMS не поддерживаются.",
    "answerable": True,
    "expected_section": "05"
  },
  {
    "id": 8,
    "user_input": "За сколько должны подтвердить обращение P1 и какое целевое время его решения?",
    "reference": "Подтверждение P1 — в течение 10 минут; целевое время решения — 4 часа.",
    "answerable": True,
    "expected_section": "06"
  },
  {
    "id": 9,
    "user_input": "Как долго хранятся журналы действий и когда делают резервные копии?",
    "reference": "Журналы хранятся 90 дней, резервные копии базы запускаются ежедневно в 02:00 по московскому времени.",
    "answerable": True,
    "expected_section": "07"
  },
  {
    "id": 10,
    "user_input": "Какие способы экспорта данных доступны на тарифе «Старт»?",
    "reference": "На тарифе «Старт» доступна выгрузка CSV, но REST API доступен только на тарифе «Команда», а прямого XLSX-экспорта нет.",
    "answerable": True,
    "expected_section": "08"
  },
  {
    "id": 11,
    "user_input": "Каков целевой uptime и когда проходит плановое обслуживание?",
    "reference": "Целевой показатель доступности — 99,5% в месяц. Плановое обслуживание — по четвергам с 21:00 до 22:00 по Москве.",
    "answerable": True,
    "expected_section": "09"
  },
  {
    "id": 12,
    "user_input": "Сколько рабочих дней занимает подключение и сколько длится вебинар?",
    "reference": "До 3 рабочих дней после получения корректного списка устройств; вебинар длится 90 минут.",
    "answerable": True,
    "expected_section": "10"
  },
  {
    "id": 13,
    "user_input": "Сколько складов участвовало в пилоте к концу июня 2025 года?",
    "reference": "7 складов в трёх регионах; названия регионов не указаны.",
    "answerable": True,
    "expected_section": "11"
  },
  {
    "id": 14,
    "user_input": "Какие технологии используются для API, базы данных и фоновых задач?",
    "reference": "Python и FastAPI для веб-API, PostgreSQL для основной базы, Redis для очереди фоновых задач.",
    "answerable": True,
    "expected_section": "12"
  },
  {
    "id": 15,
    "user_input": "Какой телефон у технической поддержки?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 16,
    "user_input": "Как зовут генерального директора «Люмен Лаб»?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 17,
    "user_input": "Какую выручку компания получила в 2025 году?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 18,
    "user_input": "В каких именно регионах находились семь пилотных складов?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "11"
  }
]

print("Ручных эталонов:", len(BUILTIN_EXAMPLES))

In [ ]:
import pandas as pd
from pathlib import Path

TESTSET_CSV = Path("ragas_generated_testset.csv")

if TESTSET_MODE == "example":
    generated_df = pd.DataFrame(BUILTIN_EXAMPLES)
    print("Встроенный набор: вопросов", len(generated_df))

elif TESTSET_MODE == "upload":
    # utf-8-sig позволяет корректно прочитать CSV с русскими буквами из Excel/Colab.
    generated_df = pd.read_csv(UPLOAD_CSV_PATH, encoding="utf-8-sig", keep_default_na=False)
    if "question" in generated_df and "user_input" not in generated_df:
        generated_df = generated_df.rename(columns={"question": "user_input"})
    print("Загружено вопросов:", len(generated_df))

elif TESTSET_MODE == "generate":
    if TESTSET_CSV.exists() and not REGENERATE_TESTSET:
        generated_df = pd.read_csv(TESTSET_CSV, encoding="utf-8-sig")
        print("Загружен сохранённый набор:", len(generated_df))
    else:
        from langchain_core.documents import Document
        from ragas.testset import TestsetGenerator
        from ragas.testset.persona import Persona
        from ragas.testset.transforms.extractors.llm_based import NERExtractor
        from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
        from ragas.testset.synthesizers.prompts import ThemesPersonasMatchingPrompt, PersonaThemesMapping
        from ragas.run_config import RunConfig

        # RAGAS иногда переводит имя персоны, что вызывает KeyError при поиске по имени.
        # Здесь одна персона: темы привязываем к ней детерминированно.
        class ExactPersonaMatchingPrompt(ThemesPersonasMatchingPrompt):
            async def generate(self, data, llm=None, callbacks=None, **kwargs):
                return PersonaThemesMapping(mapping={
                    persona.name: list(data.themes) for persona in data.personas
                })

        source_docs = [Document(page_content=text,
                                metadata={"source": str(TXT_PATH), "section": i+1})
                       for i, text in enumerate(chunks)]
        persona = Persona(name="Сотрудник компании",
                          role_description="Задаёт вопросы по справочнику, тарифам и процессам")
        generator = TestsetGenerator(llm=generator_ragas_llm,
                                     embedding_model=ragas_embeddings,
                                     persona_list=[persona])
        synthesizer = SingleHopSpecificQuerySynthesizer(llm=generator_ragas_llm)
        # Адаптация на русский язык полезна, но для скорости выключена.
        ADAPT_RUSSIAN = False
        if ADAPT_RUSSIAN:
            ru_prompts = await synthesizer.adapt_prompts("russian", llm=generator_ragas_llm)
            synthesizer.set_prompts(**ru_prompts)
        synthesizer.theme_persona_matching_prompt = ExactPersonaMatchingPrompt()

        # Только GENERATOR, а не ANSWER и не JUDGE, составляет вопросы и эталоны.
        testset = generator.generate_with_langchain_docs(
            source_docs, testset_size=TESTSET_SIZE,
            transforms=[NERExtractor()],
            query_distribution=[(synthesizer, 1.0)],
            run_config=RunConfig(timeout=180, max_workers=2, max_retries=2),
        )
        generated_df = testset.to_pandas().copy()
        generated_df = generated_df.dropna(subset=["user_input", "reference"]).reset_index(drop=True)
        generated_df.insert(0, "id", range(1, len(generated_df)+1))
        generated_df.to_csv(TESTSET_CSV, index=False, encoding="utf-8-sig")
        print("RAGAS сгенерировал и сохранил", len(generated_df), "тестов")
else:
    raise ValueError("TESTSET_MODE должен быть generate, upload или example")

required = {"user_input", "reference"}
if not required.issubset(generated_df.columns):
    raise ValueError(f"В тестах нужны поля {required}, получено {list(generated_df.columns)}")
if "id" not in generated_df.columns:
    generated_df.insert(0, "id", range(1, len(generated_df)+1))
generated_df["id"] = generated_df["id"].astype(str)
generated_df["user_input"] = generated_df["user_input"].astype(str).str.strip()
generated_df["reference"] = generated_df["reference"].astype(str).str.strip()
if generated_df.empty or (generated_df[["user_input", "reference"]] == "").any().any():
    raise ValueError("В CSV отсутствуют вопросы или эталонные ответы")
if generated_df["id"].duplicated().any():
    raise ValueError("В CSV найдены повторяющиеся id")

print("Итоговый набор:", len(generated_df), "вопросов")
display(generated_df[["id", "user_input", "reference"]].head(12))

## 6. Проверьте качество эталонов до эксперимента

В `upload` и `example` эталоны составлены вручную по TXT; в `generate` их подготовила LLM, поэтому **несколько примеров обязательно проверьте**. Даже сильная модель может придумать эталон, не подтверждённый документом. В режиме `generate` сохраните/отредактируйте CSV и выберите `upload`, если нужна ручная ревизия.

Отдельно в ручном наборе есть вопросы **без ответа** с эталоном `НЕТ_ДАННЫХ` — автоматический генератор RAGAS обычно не гарантирует их наличие.

In [ ]:
for row in generated_df.head(4).itertuples():
    print("\n" + "="*60)
    print("Вопрос:", row.user_input)
    print("Эталон:", row.reference)

if "answerable" in generated_df:
    print("\nВручную размеченные вопросы без ответа:",
          int(generated_df["reference"].eq("НЕТ_ДАННЫХ").sum()))
print("\nЭталон НЕ передаётся модели-ответчику.")

# Часть 2. Участники улучшают RAG-промпт

**Задача участника:** изменить только `STUDENT_RAG_PROMPT`, чтобы сократить галлюцинации. Модель получает **вопрос и извлечённый контекст**. Эталон (`reference`) используется позже исключительно для оценки и никогда не передаётся модели-ответчику.

В промпте обязательно должны оставаться `{question}` и `{context}`. Если вставляете JSON-пример, удваивайте фигурные скобки `{{` и `}}`.

In [ ]:
BASELINE_RAG_PROMPT = """
Ответь на вопрос, используя материалы ниже.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:
"""

# ✏️ САМОСТОЯТЕЛЬНОЕ ЗАДАНИЕ: измените только STUDENT_RAG_PROMPT.
# Подумайте: как не добавлять факты от себя, как давать ссылки на источник,
# когда честнее отказаться, как не пропускать важные детали?
STUDENT_RAG_PROMPT = """
Ты помощник по внутреннему справочнику компании.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

Ответ:
"""

## 7. Запускаем модель-ответчик с двумя промптами

`RUN_N=3` в верхней ячейке для быстрого теста. `RUN_N=None` — на всех доступных примерах. **Менять модель-ответчик, `E5_MODEL`, `TOP_K` или набор документов между промптами не нужно**, иначе не получится изолировать влияние промпта.

In [ ]:
import pandas as pd
from langchain_core.messages import HumanMessage

selected_tests = generated_df if RUN_N is None else generated_df.head(RUN_N)
if selected_tests.empty:
    raise ValueError("Выберите хотя бы один вопрос")

# Retrieval готовим ОДИН РАЗ, поэтому оба промпта видят идентичные документы.
retrieval_cache = {
    str(row.id): retrieve(row.user_input)
    for row in selected_tests.itertuples()
}

def run_rag(question, prompt_template, contexts):
    rendered = prompt_template.format(
        context="\n\n".join(contexts),
        question=question,
    )
    return str(answer_model.invoke([HumanMessage(content=rendered)]).content).strip()

def run_experiment(prompt_template, title):
    results = []
    for row in selected_tests.itertuples():
        contexts = retrieval_cache[str(row.id)]
        answer = run_rag(row.user_input, prompt_template, contexts)
        results.append({
            "id": str(row.id),
            "user_input": row.user_input,
            "reference": row.reference,
            "retrieved_contexts": contexts,
            "response": answer,
        })
        print(f"{title}: вопрос #{str(row.id)} готов")
    return pd.DataFrame(results)

baseline_df = run_experiment(BASELINE_RAG_PROMPT, "Базовый")
student_df = run_experiment(STUDENT_RAG_PROMPT, "Промпт участника")

preview = student_df[["id", "user_input", "response", "reference"]]
display(preview)

# Часть 3. Оценка независимым LLM-судьёй

**Важный принцип:** `judge_ragas_llm` (не `answer_model` и не `generator_ragas_llm`) передаётся в `ragas.evaluate`. Судья оценивает оба эксперимента одинаковыми метриками. Отдельно оцениваем поведение на вопросах без ответа.

Файлы с эталонами помогают обойти ошибки **генерации эталонов**, но **не исправляют автоматически слабую оценку судьи**.

## 7. Как понимать метрики

| Метрика RAGAS | Что проверяет | Лучше |
|---|---|---|
| **Faithfulness** | Какая доля проверяемых утверждений в ответе подтверждается **найденными RAG фрагментами**. **Главная метрика про галлюцинации относительно контекста**. | ↑ к 1 |
| **Factual Correctness (F1)** | Согласованность фактов ответа с **синтетическим эталоном**, баланс ошибок и пропущенных фактов. | ↑ к 1 |
| **Response Relevancy** | Насколько ответ релевантен вопросу. Не гарантирует достоверность! | ↑ к 1 |
| **Context Precision** | Насколько высоко в выдаче ранжируются релевантные куски текста. | ↑ к 1 |
| **Context Recall** | Содержит ли выдача достаточно фактов, необходимых для эталонного ответа. | ↑ к 1 |
| **Semantic Similarity** | Насколько ответ близок по смыслу к эталону (эмбеддинги). Можно быть семантически похожим и фактически ошибочным. | ↑ к 1 |
| **Noise Sensitivity (опционально)** | Насколько ответ допускает ошибки из-за посторонних контекстных фрагментов. | ↓ к 0 |

**Дополнительный показатель в этом Colab:** `1 - Faithfulness` — *приближённая доля неподтверждённых утверждений*, **не** точная вероятность галлюцинации и не доля «ответов с галлюцинациями».

Низкая Faithfulness может означать, что LLM придумала факт **или** что нужный факт просто не попал в retrieved contexts. Именно поэтому сравниваем её с Context Recall. Не усредняйте метрики вслепую: они отвечают на разные вопросы.

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    SemanticSimilarity,
    NoiseSensitivity,
)

METRICS = [
    Faithfulness(),
    FactualCorrectness(mode="f1"),
    ResponseRelevancy(strictness=1),   # экономный режим для учебной практики
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
    SemanticSimilarity(),
]
if INCLUDE_EXTRA_METRICS:
    METRICS.append(NoiseSensitivity(mode="irrelevant"))

print("Активные метрики:", [metric.name for metric in METRICS])

## 8. Запуск RAGAS на ответах обоих промптов

На каждом шаге RAGAS использует **фиксированный эталон и независимую модель JUDGE**. Если в каком-то примере метрика не рассчиталась, она будет `NaN`, а не 0. Проверьте количество валидных оценок перед сравнением средних.

In [ ]:
def evaluate_answers(experiment_df):
    records = experiment_df[
        ["user_input", "retrieved_contexts", "response", "reference"]
    ].to_dict("records")
    dataset = EvaluationDataset.from_list(records)
    result = evaluate(
        dataset=dataset,
        metrics=METRICS,
        llm=judge_ragas_llm,
        embeddings=ragas_embeddings,
        run_config=RunConfig(timeout=180, max_workers=2, max_retries=2),
        raise_exceptions=False,
        show_progress=True,
    )
    scored = result.to_pandas().reset_index(drop=True)
    scored.insert(0, "id", experiment_df["id"].to_list())
    if "faithfulness" in scored.columns:
        scored["unsupported_fraction_estimate"] = 1 - pd.to_numeric(
            scored["faithfulness"], errors="coerce"
        )
    return scored

baseline_scores = evaluate_answers(baseline_df)
student_scores = evaluate_answers(student_df)

names = [m.name for m in METRICS] + ["unsupported_fraction_estimate"]
names = [name for name in names if name in student_scores.columns]
display(student_scores[["id", "user_input"] + names])

## 9. Сводные метрики: сравниваем два промпта

Сравнивайте значения и количество **валидно посчитанных** оценок. Для большинства метрик больше — лучше; у `Noise Sensitivity` и `1 - Faithfulness` — наоборот.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rows = []
for name in names:
    before = pd.to_numeric(baseline_scores[name], errors="coerce")
    after = pd.to_numeric(student_scores[name], errors="coerce")
    rows.append({
        "metric": name,
        "baseline": before.mean(),
        "student": after.mean(),
        "change": after.mean() - before.mean(),
        "baseline_valid_n": int(before.notna().sum()),
        "student_valid_n": int(after.notna().sum()),
        "better": "↓" if name.startswith("noise_sensitivity") or name == "unsupported_fraction_estimate" else "↑",
    })
summary = pd.DataFrame(rows)
display(summary.round(3))

ax = summary.set_index("metric")[["baseline", "student"]].plot(
    kind="bar", figsize=(11, 5), rot=35
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Средний балл RAGAS (обычно 0–1)")
ax.set_title("Базовый vs свой промпт: один и тот же тестовый набор")
plt.tight_layout()
plt.show()

## 9. Отдельно оцениваем галлюцинации и необоснованные ответы

`Faithfulness` — оценка доли утверждений, поддержанных **найденными** фрагментами. `1 - Faithfulness` — удобный *индикатор* неподтверждённых утверждений, но не доказанная доля ложных фактов: это оценка LLM-судьи, и она может быть ошибочной.

На примерах с эталоном `НЕТ_ДАННЫХ` считаем **долю правильных отказов** и **долю выдуманных ответов** без участия судьи. Для автоматических тестов RAGAS таких вопросов может не быть, тогда метрики отказов будут не определены.

In [ ]:
import numpy as np

ABSTENTION = "НЕТ_ДАННЫХ"

def add_abstention_stats(df):
    gold_empty = df["reference"].astype(str).str.strip().eq(ABSTENTION)
    response_empty = df["response"].astype(str).str.strip().str.upper().eq(ABSTENTION)
    n_unanswerable = int(gold_empty.sum())
    n_answerable = int((~gold_empty).sum())
    return {
        "n_unanswerable": n_unanswerable,
        "correct_abstention_rate": (float(response_empty[gold_empty].mean()) if n_unanswerable else np.nan),
        "unsupported_answer_rate": (float((~response_empty[gold_empty]).mean()) if n_unanswerable else np.nan),
        "unnecessary_abstention_rate": (float(response_empty[~gold_empty].mean()) if n_answerable else np.nan),
    }

abstention_table = pd.DataFrame([
    {"prompt": "baseline", **add_abstention_stats(baseline_df)},
    {"prompt": "student", **add_abstention_stats(student_df)},
])
display(abstention_table.round(3))
print("Эти метрики проверяют точное слово НЕТ_ДАННЫХ — договоритесь о формате отказа в промпте.")

## 10. Ищем самые подозрительные ответы

Обратите внимание на примеры с **низкой Faithfulness** — это первый сигнал о неподтверждённых утверждениях. Посмотрите на ответ и найденные фрагменты: причина в плохом поиске или модель добавила информацию, которой не было в контексте?

In [ ]:
if "faithfulness" in student_scores.columns:
    review = student_scores.sort_values("faithfulness", na_position="last")
    for row in review.head(3).itertuples(index=False):
        print("\n" + "=" * 70)
        print("ВОПРОС:", row.user_input)
        print("ОТВЕТ RAG:", row.response)
        print("ЭТАЛОН:", row.reference)
        print("FAITHFULNESS:", row.faithfulness)
        for source in row.retrieved_contexts:
            print("  ИСТОЧНИК:", source[:240].replace("\n", " "), "...")
else:
    print("Метрика Faithfulness отсутствует. Проверьте ошибки оценки выше.")

## 10. Сохраняем результаты и собственный фиксированный набор эталонов

Сохраняем текущий тестовый набор, ответы и оценки. Если сейчас использован `generate`, CSV можно **отредактировать вручную и загрузить через `upload`** для следующего прогона. Изменённый промпт не должен изменять вопросы или эталоны в одном сравнении.

In [ ]:
generated_df.to_csv("ragas_current_references.csv", index=False, encoding="utf-8-sig")
baseline_scores.to_csv("ragas_baseline_scores.csv", index=False, encoding="utf-8-sig")
student_scores.to_csv("ragas_student_scores.csv", index=False, encoding="utf-8-sig")
summary.to_csv("ragas_summary.csv", index=False, encoding="utf-8-sig")
abstention_table.to_csv("ragas_abstention_metrics.csv", index=False, encoding="utf-8-sig")
print("Сохранены 5 CSV-файлов с тестами, оценками и метриками отказов.")

## Обсуждение после практики

1. **Что улучшилось от промпта?** Сравните Faithfulness и Factual Correctness, затем проверьте подозрительные примеры вручную.
2. **Если Context Recall низкая, кто виноват?** Промпт не может восстановить факты из плохо найденного контекста.
3. **Почему нельзя доверять единственному баллу судьи?** Даже сильный LLM-судья может ошибаться. Для серьёзной оценки часть примеров размечают люди.
4. **Что меняется, если эталоны загружаем вручную?** Убираем один источник ошибок — автоматическую генерацию эталонов, но не ошибки поиска, ответа и судьи.
5. **Вопросы без ответа.** Какие промпты помогли отвечать `НЕТ_ДАННЫХ` вместо выдуманного телефона/выручки/директора?

**Ограничение:** эксперименты с тремя моделями могут быть затратными. Для первого прогона используйте `example` и `RUN_N=3`.

5. **Что изменилось после перехода от TF-IDF к E5?** Теперь retrieval учитывает смысловую близость, но всё равно может найти нерелевантный фрагмент. Помните, что даже идеально написанный промпт не компенсирует плохой поиск.
